<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
DeepAgents - Meeting/Briefing-Skill
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

from genai_lib.utilities import (
    check_environment, get_ipinfo, mprint, mermaid, setup_api_keys, show_trace, copy_from_github,
)

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M33-Meeting-Briefing"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M33_DeepAgents_Skill_Meeting_Briefing",
    "tags": ["m33", "deepagents"],
    "metadata": {"notebook": "M33", "version": "1.0"}
}


In [ ]:
#@title 📦 Installationen { display-mode: "form" }

!uv pip install --system -q "deepagents==0.6.1"
!uv pip install --system -q markitdown[all]

# 1 | Übersicht

---

<div style="background-color:#1a1a2e; padding:30px; border-radius:12px; margin-bottom:10px">
  <h1 style="color:#e0e0e0; font-family:monospace; margin:0">Meeting-Briefing Skill mit DeepAgents</h1>
  <p style="color:#a0a0c0; font-family:monospace; margin:8px 0 0 0">Skill-Dateien aus GitHub · Gate-Analyse mit o3 · structured_output · Writer Sub-Agent</p>
</div>

**Was dieses Notebook zeigt:**
- Skill-Dateien (`SKILL.md`, `WRITER.md`, Regelwerke) **direkt aus dem GitHub-Repo laden**
- Gate-Analyse mit `o3` + `with_structured_output()` → deterministisches JSON
- `extract_actions.py` aus GitHub holen, lokal zwischenspeichern, deterministisch aufrufen
- **Writer als Sub-Agent** mit geladenem `WRITER.md` als System-Prompt
- DeepAgent-Koordinator orchestriert Skill-Abarbeitung autonom

**Verwandte Module:** M31 – Agent Skill Compliance · M32 – DeepAgents Harness

---

<font color="black" size="5">Was dieses Notebook zeigt</font>

In `06_skill/meeting-briefing` im GitHub-Repo liegt ein vollständiger Skill:
Kontext laden, Agenda nach festen Regeln strukturieren, Action Items deterministisch
extrahieren und das Ergebnis im definierten Briefing-Format ausgeben.

Dieses Notebook baut daraus einen **DeepAgent** mit vier Kernmerkmalen:

| # | Merkmal | Warum |
|---|---------|-------|
| 1 | **Skill-Dateien aus GitHub laden** | Änderungen am Skill greifen ohne Notebook-Anpassung |
| 2 | **`with_structured_output()`** | Gate-Analyse gibt Pydantic-validiertes JSON zurück |
| 3 | **Modell-Auswahl laut Guide** | `o3` für Koordinator & Gate-Analyse |
| 4 | **Writer als Sub-Agent** | Trennung von Analyse und Formatierung |

> **Best Practice:** DeepAgents ist hier die Harness-Schicht, nicht die Wissensquelle.
> Die Fachlogik bleibt im Skill-Repo, in den Regelwerken und im deterministischen Tooling verankert.

In [ ]:
#@markdown   <p><font size="4" color="green">Skill-Architektur</font></p>

diagram_arch = '''
%%{init: {'theme':'light'}}%%
flowchart TB
    GH([" 🐙 GitHub Repo\nralf-42/Agenten"]) --> LOAD
    U([" 🧑 Meeting-Anfrage"]) --> K

    subgraph K_BLOCK [" 🤖 DeepAgent Koordinator — o3"]
        LOAD["Skill-Dateien laden\n(copy_from_github)"] --> K
        K["Kontext laden\nAction Items extrahieren"]
        K --> G["🛠️ analysiere_meeting_kontext\no3 + with_structured_output"]
        K --> E["🛠️ extract_action_items\nextract_actions.py als Subprocess"]
        G --> JSON["GateOutput JSON\nPydantic-validiert"]
        E --> JSON
        JSON --> W["Sub-Agent\nbriefing-writer"]
    end

    W --> B([" ✅ Meeting-Briefing"])

    style GH   fill:#24292e,stroke:#555,color:#fff
    style U    fill:#10a37f,stroke:#333,color:#000
    style B    fill:#10a37f,stroke:#333,color:#000
    style G    fill:#87CEEB,stroke:#333,color:#000
    style E    fill:#87CEEB,stroke:#333,color:#000
    style JSON fill:#FFD700,stroke:#333,color:#000
    style W    fill:#FFA500,stroke:#333,color:#000
    style LOAD fill:#6e40c9,stroke:#333,color:#fff
'''
mermaid(diagram_arch, width=580)

# 2 | Skill-Dateien aus GitHub laden

---



Alle Skill-Dateien werden **einmalig aus dem GitHub-Repo** in ein lokales Cache-Verzeichnis geladen:

| Datei | Zweck |
|-------|-------|
| `SKILL.md` | Gate-Regeln, Hard Rules, Workflow-Definition |
| `references/writer-format.md` | Anweisungen für den Writer Sub-Agenten |
| `references/agenda_rules.md` | Prioritäten, Agenda-Aufbau |
| `references/action_rules.md` | Action-Item-Erkennungsregeln |
| `references/examples.md` | Referenzbeispiele für Gate und Writer |
| `scripts/extract_actions.py` | Deterministisches Skript — direkt im Cache-Verzeichnis |

> 💡 Das Skript `extract_actions.py` liegt nach dem Download direkt im Cache-Verzeichnis
> und wird per `subprocess` aufgerufen — kein Temp-Dateien-Schreiben nötig.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path
from textwrap import dedent
from typing import List, Optional

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from markitdown import MarkItDown
from pydantic import BaseModel, Field
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver


**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

In [ ]:
# 2.1 Skill-Dateien aus GitHub laden und lokal cachen

SKILL_DIR = Path("./_skill_meeting_briefing")

copy_from_github(
    source="ralf-42/Agenten/06_skill/meeting-briefing",
    target=str(SKILL_DIR),
)

SCRIPT_PATH = SKILL_DIR / "scripts" / "extract_actions.py"

_file_map = {
    "skill":        SKILL_DIR / "SKILL.md",
    "writer":       SKILL_DIR / "references" / "writer-format.md",
    "agenda_rules": SKILL_DIR / "references" / "agenda_rules.md",
    "action_rules": SKILL_DIR / "references" / "action_rules.md",
    "examples":     SKILL_DIR / "references" / "examples.md",
}

def read_skill(name: str) -> str:
    """Liest eine Skill-Datei aus dem lokalen Cache."""
    if name not in _file_map:
        return f"Unbekannte Datei: {name}. Verfügbar: {list(_file_map.keys())}"
    return _file_map[name].read_text(encoding="utf-8").strip()

mprint(f"Skill-Verzeichnis: `{SKILL_DIR}`")
mprint(f"Skill-Dateien: {list(_file_map.keys())}")
mprint(f"Skript: `{SCRIPT_PATH}`")

In [ ]:
# 2.2 Demo-Kontext-Dokumente aus GitHub laden und lokal cachen

DATA_DIR = Path("./_data_meeting_briefing")
KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text"
KORPUS_MASKE = "0*.docx"
KORPUS_TARGET = str(DATA_DIR)

copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)

_md = MarkItDown()

DEMO_CONTEXTS_VORB = {
    p.name: _md.convert(str(p)).text_content
    for p in sorted(DATA_DIR.glob("0*.docx"))
}

# Nachbereitung: Kundengespräch (inline)
DEMO_CONTEXTS_NACH = {
    "gespraechsprotokoll.md": dedent("""
        Gesprächsprotokoll — Abstimmung Pilotprojekt Firma XYZ, 2026-03-19

        Entschieden: Pilotprojekt startet 01.04.2026, sofern Datenschutzfreigabe bis 28.03.2026 vorliegt.
        Frau Keller verantwortet die Freigabe. Herr Braun liefert bis 25.03.2026 die finale Schnittstellenliste.
        Offener Punkt: Budgetrahmen für externe Beratung ist noch nicht bestätigt.

        Nächster Termin: 26.03.2026, 10:00 Uhr.
    """)
}

mprint(f"Vorkontext-Dokumente geladen: `{len(DEMO_CONTEXTS_VORB)}`")


In [ ]:
from genai_lib.utilities import load_prompt

# Auszüge aus den lokalen Skill-Dateien (Frontmatter wird automatisch entfernt)
_previews = {
    "SKILL.md":        (str(SKILL_DIR / "SKILL.md"),                        600),
    "agenda_rules.md": (str(SKILL_DIR / "references" / "agenda_rules.md"),  400),
    "WRITER.md":       (str(SKILL_DIR / "references" / "writer-format.md"), 300),
}

zeilen = []
for name, (path, limit) in _previews.items():
    text = load_prompt(path, mode="S")[:limit]
    zeilen += [f"### {name} (Auszug)", "", text, ""]

mprint("\n".join(zeilen))

# 3 | Pydantic-Modelle & Gate-Analyse

---



Damit der Gate-Agent **deterministisch** strukturierten Output liefert,
verwenden wir `with_structured_output()` mit Pydantic-Modellen.

| Modell | Aufgabe |
|--------|---------|
| `ActionItemSchema` | Einzelnes Action Item (Was / Wer / Wann) |
| `AgendaPoint` | Agenda-Punkt (Priorität, Ziel, Dauer, Eigentümer) |
| `GateOutput` | Vollständiger strukturierter Gate-Output für den Writer |

> 💡 **Warum `with_structured_output()`?**
> Der Writer-Sub-Agent braucht verlässliche Eingabedaten. Ein freier LLM-Text-Output
> kann in Struktur und Vollständigkeit variieren — ein Pydantic-validiertes JSON nicht.

In [ ]:
# 3.1 Pydantic-Modelle für strukturierten Gate-Output

class ActionItemSchema(BaseModel):
    """Ein konkretes Action Item aus dem Meeting."""
    task:  str = Field(description="Aufgabenbeschreibung — kurz, aktionsorientiert")
    owner: str = Field(description="Verantwortliche Person — '[offen]' wenn unklar")
    due:   str = Field(description="Fälligkeitsdatum — '[offen]' wenn unbekannt")


class AgendaPoint(BaseModel):
    """Ein Agenda-Punkt mit Priorität, Ziel und Zeitplan."""
    priority: str = Field(description="Priorität: hoch, mittel oder gering")
    point:    str = Field(description="Bezeichnung des Agenda-Punkts")
    goal:     str = Field(description="Ziel: entscheiden, klären, informieren oder präsentieren")
    duration: int = Field(description="Geplante Dauer in Minuten")
    owner:    str = Field(description="Verantwortliche Person für diesen Punkt")


class GateOutput(BaseModel):
    """Strukturierter Gate-Output — Eingabe für den Writer-Sub-Agenten."""
    type:           str                    = Field(description="vorbereitung oder nachbereitung")
    meeting_type:   str                    = Field(description="Projektmeeting, Kundengespräch, Review, Workshop oder Sonstiges")
    topic:          str                    = Field(description="Meeting-Thema")
    date:           Optional[str]          = Field(default=None, description="Datum/Uhrzeit wenn bekannt")
    participants:   List[str]              = Field(description="Teilnehmerliste")
    agenda:         List[AgendaPoint]      = Field(description="Agenda-Punkte sortiert: hoch → mittel → gering")
    open_questions: List[str]              = Field(description="Offene Fragen vor dem Meeting")
    action_items:   List[ActionItemSchema] = Field(description="Action Items aus Kontext und Vorgesprächen")
    risks:          List[str]              = Field(description="Bekannte Risiken und Konflikte")
    status:         str                    = Field(description="ok, no_context oder conflict")
    conflict:       Optional[bool]         = Field(default=False, description="True bei widersprüchlichen Quellinformationen")
    decisions:      Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: getroffene Entscheidungen")
    open_points:    Optional[List[str]]    = Field(default=None, description="Nur Nachbereitung: vertagte Punkte")


# Gate-Constraint: o3 explizit auf Datenextraktion beschränken
_GATE_CONSTRAINT = """
## PFLICHT: Ausgabeformat

Du bist ein Datenextraktor — kein Schreiber.

- **Gib AUSSCHLIESSLICH die strukturierten Felder aus GateOutput zurück.**
- Formuliere KEIN Meeting-Briefing, keinen Report, keinen Fließtext.
- Schreibe KEINE Tabellen, KEINE Markdown-Sections, KEINEN Abschlusssatz.
- Diese Regel überschreibt alle anderen Anweisungen.
"""

# Gate-Prompt aus GitHub-Skill-Dateien zusammensetzen
_GATE_RULES = (
    read_skill("skill") + "\n\n"
    + read_skill("agenda_rules") + "\n\n"
    + read_skill("action_rules") + "\n\n"
    + _GATE_CONSTRAINT
)

gate_prompt = ChatPromptTemplate([
    ("system", _GATE_RULES),
    ("user", "Meeting-Anfrage: {anfrage}\n\nKontext-Dokumente:\n{kontext}"),
])

mprint("✅ **Pydantic-Modelle** definiert: `ActionItemSchema`, `AgendaPoint`, `GateOutput`")
mprint("✅ **Gate-Prompt** aus GitHub: `SKILL.md` + `agenda_rules.md` + `action_rules.md`")

In [ ]:
# 3.2 Gate-Analyse als @tool — o3 + with_structured_output()

@tool
def analysiere_meeting_kontext(anfrage: str, kontext: str) -> str:
    """
    Analysiert Meeting-Kontext mit o3 und gibt strukturierten Gate-Output zurück.

    Verwendet with_structured_output(GateOutput) für deterministisches,
    Pydantic-validiertes JSON. Regeln kommen aus den GitHub-Skill-Dateien.

    Args:
        anfrage: Meeting-Typ, Thema, Teilnehmer, Datum
        kontext: Gesamter Meeting-Kontext aus allen geladenen Dokumenten

    Returns:
        Pydantic-validiertes JSON (GateOutput) als String
    """
    gate_llm   = init_chat_model(JUDGE)   # kein temperature!
    gate_chain = gate_prompt | gate_llm.with_structured_output(GateOutput)
    result: GateOutput = gate_chain.invoke({"anfrage": anfrage, "kontext": kontext}, config=run_cfg)
    return result.model_dump_json(indent=2)


mprint("✅ **`analysiere_meeting_kontext`** — o3 + `with_structured_output(GateOutput)`")

**Was passiert hier?**

1. `with_structured_output(...)` — bindet das Pydantic-Schema ans Modell und erzwingt strukturierte Ausgabe

# 4 | Skill-Tools kapseln

---



Der Koordinator bekommt sechs fokussierte Tools:

| Tool | Quelle | Aufgabe |
|------|--------|---------|
| `list_context_documents` | Notebook | Verfügbare Demo-Dokumente anzeigen |
| `load_context_document` | Notebook | Einzelnes Kontextdokument laden |
| `load_skill_asset` | **GitHub** | Skill-Regelwerk zur Laufzeit nachladen |
| `extract_action_items` | **GitHub** (Temp) | Deterministisch via `extract_actions.py` |
| `analysiere_meeting_kontext` | LLM-Chain | o3 + `with_structured_output(GateOutput)` |
| `meeting_request` | Notebook | Demo-Anfrage für Beispiel 1 |

In [ ]:
# 4.1 Tool-Definitionen

@tool
def list_context_documents(typ: str = "vorbereitung") -> str:
    """Listet verfügbare Demo-Kontextdokumente für den angegebenen Meeting-Typ.

    Args:
        typ: 'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_VORB if typ == "vorbereitung" else DEMO_CONTEXTS_NACH
    return json.dumps(sorted(docs.keys()), ensure_ascii=False)


@tool
def load_context_document(name: str, typ: str = "vorbereitung") -> str:
    """Lädt ein einzelnes Demo-Kontextdokument.

    Args:
        name: Dateiname (aus list_context_documents)
        typ:  'vorbereitung' oder 'nachbereitung'
    """
    docs = DEMO_CONTEXTS_VORB if typ == "vorbereitung" else DEMO_CONTEXTS_NACH
    if name not in docs:
        return f"Dokument '{name}' nicht gefunden. Verfügbar: {sorted(docs)}"
    return docs[name]


@tool
def load_skill_asset(name: str) -> str:
    """Lädt eine Skill-Datei aus dem GitHub-Repo (SKILL.md, WRITER.md, Regelwerke, Beispiele).

    Erlaubte Werte: skill, writer, agenda_rules, action_rules, examples

    Quelle: github.com/ralf-42/Agenten — 06_skill/meeting-briefing/
    """
    key = name.strip().lower()
    if key not in SKILL_URLS:
        return f"Unbekanntes Asset '{name}'. Verfügbar: {list(SKILL_URLS.keys())}"
    return read_skill(key)   # aus Cache oder frisch von GitHub


@tool
def extract_action_items(text: str) -> str:
    """Extrahiert Action Items deterministisch über extract_actions.py (aus GitHub, lokal gecacht).

    Erkennt Aktionsverben, Eigentümer und Fälligkeitsdaten.
    Gibt JSON mit top 5 priorisierten Action Items + Backlog zurück.
    """
    payload = json.dumps({"text": text}, ensure_ascii=False)
    proc = subprocess.run(
        [sys.executable, str(SCRIPT_PATH), payload],
        capture_output=True, text=True, encoding="utf-8", check=False,
    )
    if proc.returncode != 0:
        return json.dumps({
            "status": "error",
            "stderr": proc.stderr.strip(),
            "stdout": proc.stdout.strip(),
        }, ensure_ascii=False, indent=2)
    return proc.stdout.strip()


@tool
def meeting_request() -> str:
    """Liefert die Demo-Meeting-Anfrage für Beispiel 1 (Vorbereitung Sprint-Review)."""
    return dedent("""
        Thema: Sprint-Review Meeting-Briefing Skill Modul M36
        Typ: Vorbereitung
        Datum/Zeit: 2026-03-27 10:00
        Teilnehmer: Lea (Produkt), Murat (Engineering), Sophie (Training), Jonas (QA), Klara (Moderation)
        Dauer: 30 Minuten
        Ziel: Meeting-Briefing fuer interne Review-Runde vorbereiten.
    """).strip()


ALLE_TOOLS = [
    meeting_request,
    list_context_documents,
    load_context_document,
    load_skill_asset,
    extract_action_items,
    analysiere_meeting_kontext,
]

mprint(f"✅ **{len(ALLE_TOOLS)} Tools** registriert: {[t.name for t in ALLE_TOOLS]}")

In [ ]:
# 4.2 extract_action_items — Smoke-Test
kombinierter_kontext = "\n\n".join(
    [f"## {name}\n{text}" for name, text in DEMO_CONTEXTS_VORB.items()]
)

raw = extract_action_items.invoke({"text": kombinierter_kontext}, config=run_cfg)
result = json.loads(raw)

zeilen = [
    "### 🔍 Extrahierte Action Items (Vorbereitung-Kontext)",
    "",
    "| # | Aufgabe | Verantwortlich | Fällig |",
    "|---|---------|----------------|--------|",
]
for i, item in enumerate(result.get("action_items", []), 1):
    zeilen.append(f'| {i} | {item["task"]} | {item["owner"]} | {item["due"]} |')
if result.get("backlog_count", 0) > 0:
    zeilen.append(f'\n> ⚠️ **Backlog:** {result["backlog_count"]} weitere Item(s)')
mprint("\n".join(zeilen))

<font color="black" size="5">Best-Practice-Hinweise für dieses Beispiel</font>

- **Version pinnen:** `deepagents==0.6.1` — stabile Basis für `subagents=[...]`-Syntax und HarnessProfile.
- **Skill-Dateien aus GitHub:** Regeländerungen in `SKILL.md` wirken beim nächsten Notebook-Run
  automatisch — kein manuelles Copy-Paste in Prompts.
- **`references/writer-format.md` laden:** `load_prompt(url, mode="S")` entfernt automatisch ein etwaiges Frontmatter.
  Konsistent mit M31 — kein notebook-spezifisches `read_skill()` nötig.
- **Skript als Temp-Datei:** `extract_actions.py` wird beim Setup in `tempfile.gettempdir()`
  geschrieben. Kein permanentes File-Schreiben — sauber für Cloud-Umgebungen.
- **`recursion_limit` großzügig setzen:** Planning, Tool-Calls und Sub-Agent-Delegation
  zählen alle als eigene Schritte.
- **Modell pro Sub-Agent:** Das `model`-Feld im Sub-Agent-Dict wird unterstützt.
  Writer läuft auf `gpt-5.4-mini` (WORKER), Koordinator auf `o3-mini` (ROUTER) — explizit im Code.

# 5 | DeepAgent aufbauen

---

In [ ]:
# 5.1 Writer Sub-Agent — System-Prompt via load_prompt() aus lokalem Cache (references/writer-format.md)
writer_subagent = {
    "name": "briefing-writer",
    "description": "Formatiert den Gate-Output exakt im Ausgabeformat des Meeting-Briefing-Skills.",
    "model": init_chat_model(WORKER),   # gpt-5.4-mini — Textqualität für Briefings
    "system_prompt": (
        load_prompt(str(SKILL_DIR / "references" / "writer-format.md"), mode="S")   # lokal aus Cache
        + "\n\n"
        + "Gib ausschließlich das fertige Briefing aus.\n"
        + "Keine Websuche, keine neuen Fakten, keine Spekulation."
    ),
    "tools": [],
}

# 5.2 DeepAgent Koordinator
KOORDINATOR_PROMPT = (
    "Du bist der Koordinator fuer den meeting-briefing Skill.\n\n"
    "Pflicht-Ablauf:\n"
    "1. meeting_request aufrufen (Beispiel 1) ODER Anfrage aus Nachricht lesen\n"
    "2. list_context_documents aufrufen und ALLE Dokumente mit load_context_document laden\n"
    "3. extract_action_items fuer den gesamten geladenen Kontext aufrufen\n"
    "4. analysiere_meeting_kontext aufrufen → GateOutput JSON erhalten\n"
    "5. GateOutput an briefing-writer delegieren zur finalen Formatierung\n\n"
    "Hard Rules:\n"
    "- IMMER extract_action_items aufrufen — nie Action Items frei erfinden\n"
    "- IMMER analysiere_meeting_kontext aufrufen — kein freies Briefing-Schreiben\n"
    "- Fehlende Angaben als [offen] markieren, nie spekulieren\n"
    "- Antwort auf Deutsch"
)

meeting_agent = create_deep_agent(
    model=init_chat_model(ROUTER),   # Koordinator → o3-mini (Orchestrator, kein kritischer Judge)
    tools=ALLE_TOOLS,
    subagents=[writer_subagent],
    system_prompt=KOORDINATOR_PROMPT,
    checkpointer=InMemorySaver(),
)

mprint("✅ **DeepAgent** erstellt")
mprint(f"   Koordinator: `o3-mini`  ·  Sub-Agent: `briefing-writer`  ·  Tools: {len(ALLE_TOOLS)}")

**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

<p><font color='black' size="5">
5.3 HarnessProfile — Modell-spezifische Optimierung
</font></p>

DeepAgents liefert **eingebaute Profile** für OpenAI und Anthropic.
Sie aktivieren automatisch und verbessern die Agent-Qualität laut Benchmark um 10–20 Punkte — ohne Code-Änderungen.

**Wann ein eigenes `HarnessProfile` sinnvoll ist:**

| Anwendungsfall | Beispiel |
|---------------|---------|
| Sprachvorgabe | Deutsch erzwingen, auch wenn Skill-Dateien auf Englisch sind |
| Skill-weiter Prompt-Suffix | Ausgabeformat für alle Agenten in diesem Notebook einheitlich |
| Provider-Override | Unterschiedliche Einstellungen für OpenAI vs. Anthropic |

Das Profil wird **einmalig registriert** (`register_harness_profile()`) und wirkt auf alle nachfolgenden `create_deep_agent()`-Aufrufe in dieser Session.

In [ ]:
# 5.3 HarnessProfile — einmalig registrieren, wirkt auf alle nachfolgenden create_deep_agent()-Aufrufe

from deepagents.profiles import HarnessProfile, ProviderProfile, register_harness_profile

kurs_profil = HarnessProfile(
    provider=ProviderProfile.OPENAI,
    system_prompt_suffix=(
        "Antworte ausschließlich auf Deutsch. "
        "Nutze Fachbegriffe korrekt und halte das Ausgabeformat des Skills ein."
    ),
)
register_harness_profile(kurs_profil)

mprint("✅ **HarnessProfile** registriert — OpenAI-Profil + Deutsch-Suffix")
mprint("   Eingebaute OpenAI/Anthropic-Profile: +10–20 Punkte laut Benchmark, automatisch aktiv")
mprint("   Eigenes Profil: wirkt auf alle nachfolgenden create_deep_agent()-Aufrufe")
mprint("")
mprint("> ℹ️ Zum Aktivieren für `meeting_agent`: Zelle 5.2 erneut ausführen (Rebuild des Agenten).")

In [ ]:
#@markdown   <p><font size="4" color='green'>  Ablauf im Koordinator</font> </br></p>


# Sequenzdiagramm: Ablauf im Koordinator
diagram_seq = '''
%%{init: {'theme':'light'}}%%
sequenceDiagram
    autonumber
    actor N as Nutzer
    participant K as Koordinator (o3)
    participant GH as GitHub Repo
    participant D as Dokument-Tools
    participant G as analysiere_meeting_kontext
    participant E as extract_action_items
    participant W as briefing-writer

    Note over K,GH: Setup (einmalig)
    GH-->>K: SKILL.md, WRITER.md, Regelwerke, extract_actions.py

    N->>K: Meeting-Anfrage
    K->>D: list_context_documents()
    D-->>K: ["doc1.md", "doc2.txt", ...]
    K->>D: load_context_document(name) [x n]
    D-->>K: Dokumentinhalte
    K->>E: extract_action_items(kontext)
    E-->>K: Action Items JSON
    K->>G: analysiere_meeting_kontext(anfrage, kontext)
    G-->>K: GateOutput JSON (Pydantic-validiert)
    K->>W: GateOutput übergeben
    W-->>K: Formatiertes Briefing
    K-->>N: Meeting-Briefing
'''
mermaid(diagram_seq, width=1000)

In [ ]:
# -- VIZ --
from IPython.display import Image, display

display(Image(meeting_agent.get_graph(xray=True).draw_mermaid_png()))

# 6 | Beispiel-Runs

---

| Beispiel | Typ | Szenario |
|----------|-----|----------|
| 1️⃣ | Vorbereitung | Sprint-Review Modul M36 — via Demo-Dokument-Tools |
| 2️⃣ | Nachbereitung | Kundengespräch Pilotprojekt XYZ — Kontext direkt übergeben |

In [ ]:
import re as _re

def invoke_with_retry(agent, inputs, config, max_retries: int = 5):
    """Wiederholt agent.invoke() bei RateLimitError mit automatischer Wartezeit."""
    for attempt in range(1, max_retries + 1):
        try:
            return agent.invoke(inputs, config=config)
        except Exception as e:
            msg = str(e)
            if "rate_limit_exceeded" not in msg and "429" not in msg:
                raise
            match = _re.search(r"try again in ([\d.]+)s", msg)
            wait = float(match.group(1)) + 2 if match else 15
            print(f"Rate limit (Versuch {attempt}/{max_retries}) — warte {wait:.0f}s …")
            time.sleep(wait)
    raise RuntimeError(f"Rate limit nach {max_retries} Versuchen nicht behoben.")

In [ ]:
# 6.1 Beispiel: Sprint-Review Vorbereitung
aufgabe_1 = dedent("""
    Bereite das Meeting vor und wende den meeting-briefing Skill korrekt an.
    Nutze alle verfuegbaren Demo-Dokumente (typ='vorbereitung'),
    extrahiere Action Items mit dem Tool und gib am Ende nur das fertige Briefing aus.
""").strip()

print(aufgabe_1)
print()

t0 = time.perf_counter()
result_1 = invoke_with_retry(
    meeting_agent,
    {"messages": [{"role": "user", "content": aufgabe_1}]},
    config={
        "recursion_limit": 120,
        "run_name": "m33-briefing-vorbereitung",
        "tags": ["m36", "deepagents", "skill", "vorbereitung"],
    },
)
latenz_1 = round((time.perf_counter() - t0) * 1000)

mprint("\n".join([
    "## Laufmetriken Beispiel 1",
    "",
    "| Metrik | Wert |",
    "|--------|------|",
    f"| Nachrichten gesamt | `{len(result_1['messages'])}` |",
    f"| Latenz             | `{latenz_1} ms` |",
    f"| Letzte Rolle       | `{result_1['messages'][-1].type}` |",
]))

In [ ]:
briefing_1 = result_1["messages"][-1].content
mprint(briefing_1)

In [ ]:
# 6.2 Beispiel: Kundengespräch Nachbereitung
kontext_nach = "\n\n".join(
    [f"## {name}\n{text}" for name, text in DEMO_CONTEXTS_NACH.items()]
)

aufgabe_2 = dedent(f"""
    Erstelle die Nachbereitung fuer das folgende Kundengespräch.
    Wende den meeting-briefing Skill korrekt an und extrahiere
    Action Items mit dem Tool. Gib am Ende nur das fertige Briefing aus.

    Meeting-Anfrage:
    - Thema: Abstimmung Pilotprojekt Firma XYZ
    - Datum: 2026-03-19
    - Teilnehmer: Max (intern), Dr. Mueller (Kunde), Petra (Vertrieb)
    - Typ: Nachbereitung

    Kontextdokumente:
    {kontext_nach}
""").strip()

t0 = time.perf_counter()
result_2 = invoke_with_retry(
    meeting_agent,
    {"messages": [{"role": "user", "content": aufgabe_2}]},
    config={
        "recursion_limit": 120,
        "run_name": "m33-briefing-nachbereitung",
        "tags": ["m36", "deepagents", "skill", "nachbereitung"],
    },
)
latenz_2 = round((time.perf_counter() - t0) * 1000)

mprint("\n".join([
    "## Laufmetriken Beispiel 2",
    "",
    "| Metrik | Wert |",
    "|--------|------|",
    f"| Nachrichten gesamt | `{len(result_2['messages'])}` |",
    f"| Latenz             | `{latenz_2} ms` |",
    f"| Letzte Rolle       | `{result_2['messages'][-1].type}` |",
]))

briefing_2 = result_2["messages"][-1].content
mprint(briefing_2)

In [ ]:
#@markdown   <p><font size="4" color="green">Trace-Analyse</font></p>

time.sleep(2)
show_trace("M33-Meeting-Briefing", limit=3, show_steps=True)

# 7 | Einordnung

---

<font color="black" size="5">Was an diesem Beispiel wichtig ist</font>

**Skill-Dateien aus dem Repo — nicht im Notebook**

Prompts, Regeln und Skripte kommen aus `github.com/ralf-42/Agenten`.
Das Notebook ist der Ausführungskontext, nicht die Wissensquelle.
Ändert sich `SKILL.md` im Repo, übernimmt das Notebook die neue Version
beim nächsten Run automatisch — ohne manuelles Copy-Paste.

**Structured Output schützt den Writer**

`with_structured_output(GateOutput)` stellt sicher, dass der Writer-Sub-Agent
immer vollständige, typisierte Eingaben erhält — unabhängig davon, wie der
Koordinator die Analyse formuliert.

**Grenzen des Musters**

| Grenze | Erklärung |
|--------|-----------|
| Sub-Agent erbt Modell | Writer nutzt `o3` statt idealerweise `gpt-5.1` (DeepAgents-Limit) |
| Kein HITL | Für Produktion: `interrupt()` vor Briefing-Versand ergänzen (→ M17) |
| Kein Judge | Skill-Compliance nicht automatisch geprüft (→ M31) |
| Kein Auth | GitHub-Zugriff anonym — für private Repos: Token-Authentifizierung ergänzen |

# 8 | Skills für Arbeitsstandards
---


Ein verbreitetes Missverständnis: Skills müssen keine Algorithmen enthalten. Sie können **Stil, Regeln und Qualitätsvorgaben operationalisieren** — alles, was sonst in langen System-Prompts oder unstrukturierten Dokumenten verstreut ist.

Das Meeting-Briefing in diesem Modul ist selbst ein Beispiel dafür: Es enthält keine Berechnungen, sondern **Arbeitsstandards**:
- Feste Abschnitte (Agenda, Vorabinfo, Offene Punkte, Nächste Schritte)
- Pflichtfelder mit `[offen]`-Markierung statt Weglassen
- Quellenpflicht für jede Aussage
- Ton- und Formatvorgaben über `writer-format.md`

**Weitere Anwendungsfälle für "Standards-Skills":**

| Typ | Was der Skill operationalisiert | Beispiel |
|-----|--------------------------------|---------|
| Brand Skill | Farben, Typografie, Logoregeln | CraftedWell-Stil für PPT/DOCX |
| Schreibstil | Ton, Formulierungsverbote, Zielgruppe | "Immer Du, keine Passivkonstruktionen" |
| Qualitätsprüfung | Checklisten, Mindestanforderungen | Review-Kriterien vor Veröffentlichung |
| Onboarding-Prozess | Reihenfolge, Pflichtschritte, Eskalation | Lieferanten-Compliance-Check (M31) |
| Genehmigungsworkflow | Schwellenwerte, Eskalationsstufen | Budgetfreigabe über bestimmtem Betrag |

> 💡 **Kerngedanke:** Ein Skill macht implizites Organisationswissen explizit — und damit für einen Agenten nutzbar. Nicht "was kann das Modell", sondern "was weiß unsere Organisation und wie wird es angewendet".


# 9 | Skill-Qualität und Skill-Tests
---


Wie prüft man, ob ein Skill zuverlässig funktioniert? Die Antwort ist analog zu Unit-Tests in der Softwareentwicklung: **kontrollierte Eingaben mit erwartbarem Verhalten**.

### Grundstruktur eines Skill-Tests

```
tests/
├── test_cases.yaml          # Testfälle mit Eingabe + expected_behavior
├── input/                   # Beispieldokumente als Eingabe
│   ├── sprint_review.md
│   └── kundengespräch.txt
└── expected/                # Referenzausgaben (optional, für strukturierte Formate)
    └── briefing_example.md
```

Ein Testfall (YAML-Format):

```yaml
- name: "Sprint-Review mit vollständigen Infos"
  skill: meeting-briefing
  input_files:
    - input/sprint_review.md
  prompt: "Bereite das Meeting-Briefing für diesen Sprint-Review vor."
  expected_behavior: |
    - Abschnitt 'Agenda' vorhanden
    - Abschnitt 'Offene Punkte' vorhanden
    - Keine Spekulationen; fehlende Infos als [offen] markiert
    - Maximal 300 Wörter

- name: "Fehlende Teilnehmerinfos"
  skill: meeting-briefing
  input_files:
    - input/kundengespräch.txt
  prompt: "Erstelle das Briefing für das Kundengespräch."
  expected_behavior: |
    - Teilnehmer-Abschnitt enthält [offen]-Marker
    - Kein Erfinden von Namen oder Rollen
```

### Drei Qualitätsdimensionen

| Dimension | Frage | Test-Ansatz |
|-----------|-------|-------------|
| **Vollständigkeit** | Sind alle Pflichtabschnitte vorhanden? | Regex / Strukturprüfung |
| **Korrektheit** | Werden nur Fakten aus dem Input verwendet? | LLM-as-Judge (M24-Pattern) |
| **Konsistenz** | Liefert der Skill bei gleicher Eingabe ähnliche Outputs? | Mehrfachausführung + Vergleich |

### Wann sind Skill-Tests besonders wichtig?

- Bei Skills mit **Guardrails** (was darf nicht weggelassen werden)
- Bei **mehrstufigen Workflows** mit Pflichtschritten
- Nach jeder Änderung an `SKILL.md` oder `references/`
- Vor dem **Einsetzen in Production** (M35)

> ⚠️ **Faustregel:** Wenn ein Skill in einer realen Entscheidung (Compliance, Freigabe, Kundenkommunikation) eingesetzt wird — dann braucht er Tests, bevor er deployed wird.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Skill-Test Qualitäts-Dimensionen</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    IN["Eingabe
(Dokumente + Prompt)"]
    SK["Skill ausführen
(Agent + SKILL.md)"]
    OUT["Output
(Briefing / Ergebnis)"]
    subgraph Checks ["Qualitätsprüfung"]
        C1["Vollständigkeit
Pflichtabschnitte vorhanden?"]
        C2["Korrektheit
Nur Fakten aus Input?"]
        C3["Konsistenz
Wiederholbar bei gleicher Eingabe?"]
    end
    IN --> SK --> OUT --> Checks
    C1 -->|"✅ / ❌"| Report["Test-Report"]
    C2 -->|"✅ / ❌"| Report
    C3 -->|"✅ / ❌"| Report
    style C1 fill:#2E7D32,color:#fff
    style C2 fill:#1565C0,color:#fff
    style C3 fill:#E65100,color:#fff
'''
mermaid(diagram, width=650)


# A | Aufgabe

---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf und soll im Kurs auch als Lernunterstützung genutzt werden — z. B. Gemini in Google Colab, um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.

**Grundlagen**

Erstelle ein Briefing für ein eigenes Meeting-Szenario: Füge ein neues Demo-Dokument hinzu und passe die Agenda an. Das Briefing soll alle Pflichtfelder enthalten.

**✅ Erledigt wenn:** Das Briefing enthält alle Pflichtfelder aus dem Schema — keine leeren Sektionen.

In [ ]:
# Grundlagen: Eigenes Meeting-Briefing
# Startpunkt: run_skill()-Aufruf aus Kapitel 3

mein_meeting = {
    'titel': '...',
    'datum': '...',
    'teilnehmer': [],
    'agenda': [],
    'dokumente': [],  # eigene Demo-Dokumente
}

# Skill aufrufen:
# result = run_skill('meeting_briefing', mein_meeting)
# mprint(result['briefing'])

**Aufbau**

Teste den `no_context`-Sonderfall: Rufe `analysiere_meeting_kontext()` mit einem Dokument auf, das keinen relevanten Inhalt enthält.

**✅ Erledigt wenn:** Der `no_context`-Fall gibt eine klare Meldung zurück — kein Absturz, kein leerer Output.

In [ ]:
# Aufbau: no_context-Sonderfall testen
# Startpunkt: analysiere_meeting_kontext() aus Kapitel 4

# Dokument ohne relevanten Inhalt:
leeres_dokument = 'Dieses Dokument enthält keine relevanten Informationen.'

# result = analysiere_meeting_kontext(
#     kontext=leeres_dokument,
#     agenda='Strategiemeeting'
# )

# Prüfen: status == 'no_context'?
# print(result['status'])  # erwartet: 'no_context'
# print(result['message'])

**Vertiefung**

Baue ein Konflikt-Szenario: Zwei Dokumente mit widersprüchlichen Aussagen. Ergänze einen LCEL-Judge, der den Widerspruch erkennt und bewertet.

**✅ Erledigt wenn:** Der Judge erkennt den eingebauten Widerspruch und gibt ein strukturiertes Urteil mit Begründung aus.

In [ ]:
# Vertiefung: Konflikt-Szenario + LCEL-Judge
# Startpunkt: LLM-as-Judge aus M08/M09

from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class KonfliktUrteil(BaseModel):
    konflikt_gefunden: bool
    beschreibung: str = Field(description='Was widerspricht sich?')
    schweregrad: str = Field(description='niedrig / mittel / hoch')

# doc_a = 'Budget genehmigt: 50.000 EUR'
# doc_b = 'Budget abgelehnt in Q3-Review'

# judge_llm = init_chat_model(JUDGE)  # gpt-5.4 empfohlen
# judge = judge_llm.with_structured_output(KonfliktUrteil)
# urteil = judge.invoke(f'Dokument A: {doc_a}\nDokument B: {doc_b}', config=run_cfg)
# print(urteil)